In [64]:
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace
from IPython.display import clear_output
from astropy.table import Table, join
from pathlib import Path
from collections import Counter
import pandas as pd
import astropy.units as u
from pathlib import Path
import re

import run_config
import utils_lya_halo

from run_config import cfg, smoke
from utils_lya_halo import (run_extract, run_stack, read_galaxy_fits, measure, plotting, analysis, multicat,
core, PipelineConfig, stack, smhm)
from utils_lya_halo.io import read_galaxy_fits, apply_finite_cut
from utils_lya_halo.measure import measure_all_bins, integrated_line_flux
from utils_lya_halo.stack import build_stacks

%matplotlib inline

LYA_REST = PipelineConfig.LYA_REST

import warnings
from astropy.utils.exceptions import AstropyUserWarning

warnings.filterwarnings(
    "ignore",
    message="Input data contains invalid values.*",
    category=AstropyUserWarning,
)

In [65]:
mosdef_path = "catalogs/lya_halo_catalog_cuts/lya_mosdef_catalog_KEEP.fits"
mosdef = Table.read(mosdef_path, format='fits')

print(len(mosdef))
mosdef[:5]

522


INDEX,FIELD,CATALOG,ID,ID_V21,ID_V41,MASKNAME,SLITOBJNAME,RA,DEC,z,Z_QUAL,HA_FLUX,HA_FLUX_ERR,HA_FWHM,HA_FWHM_ERR,HB_FLUX,HB_FLUX_ERR,HB_FWHM,HB_FWHM_ERR,HG_FLUX,HG_FLUX_ERR,HG_FWHM,HG_FWHM_ERR,HD_FLUX,HD_FLUX_ERR,HD_FWHM,HD_FWHM_ERR,OII3727_FLUX,OII3727_FLUX_ERR,OII3727_FWHM,OII3727_FWHM_ERR,OII3730_FLUX,OII3730_FLUX_ERR,OII3730_FWHM,OII3730_FWHM_ERR,OIII4364_FLUX,OIII4364_FLUX_ERR,OIII4364_FWHM,OIII4364_FWHM_ERR,OIII4960_FLUX,OIII4960_FLUX_ERR,OIII4960_FWHM,OIII4960_FWHM_ERR,OIII5008_FLUX,OIII5008_FLUX_ERR,OIII5008_FWHM,OIII5008_FWHM_ERR,OI6302_FLUX,OI6302_FLUX_ERR,OI6302_FWHM,OI6302_FWHM_ERR,NII6550_FLUX,NII6550_FLUX_ERR,NII6550_FWHM,NII6550_FWHM_ERR,NII6585_FLUX,NII6585_FLUX_ERR,NII6585_FWHM,NII6585_FWHM_ERR,SII6718_FLUX,SII6718_FLUX_ERR,SII6718_FWHM,SII6718_FWHM_ERR,SII6733_FLUX,SII6733_FLUX_ERR,SII6733_FWHM,SII6733_FWHM_ERR,NEIII3870_FLUX,NEIII3870_FLUX_ERR,NEIII3870_FWHM,NEIII3870_FWHM_ERR,NEIII3969_FLUX,NEIII3969_FLUX_ERR,NEIII3969_FWHM,NEIII3969_FWHM_ERR,NIIHA,NIIHA_ERR,MASS_50,MASS_16,MASS_84,SFR_50,SFR_16,SFR_84,SSFR_50,SSFR_16,SSFR_84,AV_50,AV_16,AV_84,AGE_50,AGE_16,AGE_84,TAU_50,TAU_16,TAU_84,CHI2,ZFAST,EBV_STAR_16,EBV_STAR_50,EBV_STAR_84,HA_SN,HA_FLAG,HB_SN,HB_FLAG,HG_SN,HG_FLAG,HD_SN,HD_FLAG,OII3727_SN,OII3727_FLAG,OII3730_SN,OII3730_FLAG,OIII4364_SN,OIII4364_FLAG,OIII4960_SN,OIII4960_FLAG,OIII5008_SN,OIII5008_FLAG,OI6302_SN,OI6302_FLAG,NII6550_SN,NII6550_FLAG,NII6585_SN,NII6585_FLAG,SII6718_SN,SII6718_FLAG,SII6733_SN,SII6733_FLAG,NEIII3870_SN,NEIII3870_FLAG,NEIII3969_SN,NEIII3969_FLAG,O3,O3_ERR,O32,O32_ERR,EBV_NEB_16,EBV_NEB_50,EBV_NEB_84,FLUX_RADIUS,A_IMAGE,B_IMAGE,THETA_J2000,KRON_RADIUS,FWHM_IMAGE,CLASS_STAR,F_F160W,E_F160W,F_F814W,E_F814W,F_F606W,E_F606W,F_G,E_G,F_R,E_R,F_V,E_V,F_IRAC1,E_IRAC1,IRAC1_CONTAM,F_IRAC2,E_IRAC2,IRAC2_CONTAM,F_IRAC3,E_IRAC3,IRAC3_CONTAM,F_IRAC4,E_IRAC4,IRAC4_CONTAM,USE_PHOT,STAR_FLAG,PHOT_FLAGS,Z_SPEC,TOT_COR,SKELTON_MATCH,BA,INCL_DEG,GALFIT_FLAG_F160W,GALFIT_MAG_F160W,GALFIT_MAG_ERR_F160W,RE_F160W,RE_ERR_F160W,N_F160W,N_ERR_F160W,Q_F160W,Q_ERR_F160W,PA_F160W,PA_ERR_F160W,SN_F160W,VDW_MATCH_F160W,GALFIT_FLAG_F125W,GALFIT_MAG_F125W,GALFIT_MAG_ERR_F125W,RE_F125W,RE_ERR_F125W,N_F125W,N_ERR_F125W,Q_F125W,Q_ERR_F125W,PA_F125W,PA_ERR_F125W,SN_F125W,VDW_MATCH_F125W,RE_BEST,RE_BEST_SRC,Q_BEST,Q_BEST_SRC,RE_CIRC_BEST,MAG_G,MAG_R,MAG_V,MAG_F606W,MAG_F814W,MAG_F160W,GR_COLOR,Z_SECURE,Z_INWINDOW,BEST_LINE_SN,LINE_SN_OK,COORDS_OK,SAMPLE,HAS_DUP_0p5AS,DUP_DROP,BLEND_DROP,BLEND_MERGE,KEEP,XRAY_MATCH,XRAY_SEP_AS,XRAY_NAME,XRAY_FLUX_B,XRAY_SIG,AGN_XRAY,NIIHA_OK,BPT_O3HB,BPT_N2HA,AGN_OPT_USABLE,AGN_OPT,DONLEY_X,DONLEY_Y,AGN_IR_USABLE,AGN_IR,AGN_FLAG,CLEAN,MEX_O3HB,AGN_MEX,MEX_INTERMED,AGN_MEX_USABLE,AGN_MEX_APPLIED,VALID,EBV
int64,bytes6,bytes6,int64,int64,int64,bytes8,int64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,bool,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,f

In [66]:
sel_gal = mosdef["CLEAN"]
mosdef_gal = mosdef[sel_gal]
len(mosdef_gal)

452

In [67]:
bowman = Table.read('catalogs/bowman2021_master-catalog_no-agn.dat', format='ascii')

sel_field = ((bowman['field'] == 'cosmos') | (bowman['field'] == 'aegis'))
sel_z = ((bowman['z'] > 1.9) & (bowman['z'] < 3.5))
bowman = bowman[sel_z & sel_field]

print(len(bowman))
bowman[:5]

759


field,obj_id,ra,dec,z,OIII_FLUX,OIII_ERR,Hb_FLUX,Hb_ERR,OII_FLUX,OII_ERR,O32,O32_ERR,R23,R23_ERR,jh_mag,m140,snr_f140w,m160,snr_f160w,snr_opt,snr_uv,OIII_EW,OIII_EW_ERR,flux_radius,sfr_1_05,sfr_1_16,sfr_1_50,sfr_1_84,sfr_1_95,sfr_2_05,sfr_2_16,sfr_2_50,sfr_2_84,sfr_2_95,sfr_3_05,sfr_3_16,sfr_3_50,sfr_3_84,sfr_3_95,sfr_4_05,sfr_4_16,sfr_4_50,sfr_4_84,sfr_4_95,E(B-V)_05,E(B-V)_16,E(B-V)_50,E(B-V)_84,E(B-V)_95,$\delta$_05,$\delta$_16,$\delta$_50,$\delta$_84,$\delta$_95,$E_b$_05,$E_b$_16,$E_b$_50,$E_b$_84,$E_b$_95,Log Z_05,Log Z_16,Log Z_50,Log Z_84,Log Z_95,Log Mass_05,Log Mass_16,Log Mass_50,Log Mass_84,Log Mass_95,SFR10_05,SFR10_16,SFR10_50,SFR10_84,SFR10_95,SFR100_05,SFR100_16,SFR100_50,SFR100_84,SFR100_95,t10_05,t10_16,t10_50,t10_84,t10_95,t50_05,t50_16,t50_50,t50_84,t50_95,t90_05,t90_16,t90_50,t90_84,t90_95,stmass_l68,stmass,stmass_u68,3dhst_stmass,beta_lo,beta,beta_hi,sfr_lo,sfr,sfr_hi,A_V (UV slope),sSFR,sSFR_ERR,OIII_LUM,OIII_LUM_ERR,hlr_f814w_kpc,hlr_f814w_err_kpc,concentration_f814w,hlr_f160w_kpc,hlr_f160w_err_kpc,concentration_f160w,OIII/Hb,OIII/Hb_ERR,Hb/OIII,Hb/OIII_ERR,OIII/OII,OIII/OII_ERR,OII/OIII,OII/OIII_ERR,logLOIII_ERR,logLOIII,A_OIII,logLOIII0
str6,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
aegis,487,215.11340332,52.92957687,1.95648,3.9982914572864327e-17,5.6155778894472366e-18,1.26e-17,6.2e-18,3.32e-17,1.9800000000000002e-17,0.08073636532708753,0.26609239549288477,0.7640391565307888,0.24613981652984443,24.1746,24.02331872731869,19.41858297618916,23.940142980595283,16.271540546803447,16.271540546803447,20.805689757425426,125.86473657091894,19.295933175640492,4.562,0.553,0.778,1.18,1.4,1.477,-4.43,-3.34,-1.008,0.17,0.668,-4.628,-3.774,-1.68,-0.101,0.535,-4.268,-2.922,0.322,0.711,0.788,0.017,0.069,0.17,0.247,0.282,0.093,0.206,0.35,0.513,0.743,-0.121,0.08,0.866,2.28,3.783,-0.642,-0.52,-0.323,-0.164,-0.074,9.356,9.45,9.836,10.072,10.137,0.553,0.778,1.18,1.4,1.477,0.553,0.778,1.18,1.4,1.477,-1.042,-1.005,0.451,0.465,0.466,-1.298,-1.269,0.192,0.303,0.312,-1.997,-1.968,-1.372,0.048,0.07,9.45,9.836,10.072,9.87,0.166,-1.978,0.1688,0.6617527521945661,0.8171287202720023,0.9750124282010972,0.34598089887640465,-9.018871279727998,36.52291564623688,1.0942911344001461e+42,1.5369257505620029e+41,1.6541,0.021602477919524802,2.52682624129,2.0732,0.04077801251712657,2.58256372812,3.1732471883225655,1.6237990645754323,0.3151346052333911,0.16125919187021492,1.2043046558091663,0.7378779241291017,0.8303546741070306,0.5087586270347199,0.06099641599764772,42.03913288072014,1.2171081607015963,42.525976145000776
aegis,747,214.96040344,52.82379913,2.25112,2.935075376884422e-17,5.166331658291458e-18,-6.600000000000001e-18,4.7000000000000004e-18,4.7000000000000004e-18,7.2e-18,0.7955214010873073,0.6696796161997114,nan,0.32927646098841423,25.0893,99.0,1.0,24.814213520364643,9.691608943904175,9.691608943904175,12.005860706503556,201.27023883661718,41.06589531346293,3.599,0.292,0.457,0.661,0.86,0.987,-3.931,-1.974,0.053,0.687,0.976,-4.372,-3.298,-1.213,0.132,0.421,-4.632,-3.815,-1.609,-0.238

In [68]:
# --- HST-Hyperion HLSP catalog reader ---------------------------------------
def read_hyperion_cat(path, verbose=True):
    """Read an HST-Hyperion HLSP ASCII catalog (z-cat or el-cat).

    Handles the two things that break the standard readers:
      1. the column-name line lives inside the leading '#' block;
      2. z-cat's trailing `spec_com` column is free text containing spaces,
         so the per-row token count varies.
    """
    lines = Path(path).read_text(errors="replace").splitlines()

    hdr, data = None, []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("#"):
            hdr = s.lstrip("#").strip()          # keep overwriting -> last wins
        else:
            data.append(ln)

    if hdr is None:                              # header wasn't commented
        hdr, data = data[0], data[1:]

    names = hdr.split()
    n = len(names)

    if verbose:
        print(f"{n} columns: {names}")
        print("row token counts:",
              Counter(len(d.split()) for d in data).most_common(6))

    rows = [d.split(None, n - 1) for d in data]   # <- the whole fix
    rows = [r + [""] * (n - len(r)) for r in rows]
    df = pd.DataFrame(rows, columns=names)

    # numeric where the column really is numeric, text otherwise
    for c in df.columns:
        conv = pd.to_numeric(df[c], errors="coerce")
        nonblank = df[c].str.strip().ne("")
        if nonblank.sum() and (conv.notna() & nonblank).sum() / nonblank.sum() > 0.9:
            df[c] = conv
        else:
            df[c] = df[c].str.strip()

    if verbose:
        print(f"-> {len(df)} rows\n")
    return df

In [69]:
base = Path("catalogs")
stem = "hlsp_hst-hyperion_hst_wfc3_hyperion_f140w-f160w-g141_v1"

z  = read_hyperion_cat(base / f"{stem}_z-cat.txt")
el = read_hyperion_cat(base / f"{stem}_el-cat.txt")

12 columns: ['RA', 'DEC', 'mag', 'ID_C20', 'bf', 'zp_lp', 'zp_ez', 'ID_g', 'zg', 'qf_g', 'qf_gz', 'spec_com']
row token counts: [(12, 4537), (13, 3536), (11, 2083), (14, 1766), (15, 623), (16, 182)]
-> 12812 rows

25 columns: ['ID_g', 'RA', 'DEC', 'z_g', 'mag', 'EW50_OII', 'EW16_OII', 'EW84_OII', 'F_OII', 'eF_OII', 'EW50_Hb', 'EW16_Hb', 'EW84_Hb', 'F_Hb', 'eF_Hb', 'EW50_OIII', 'EW16_OIII', 'EW84_OIII', 'F_OIII', 'eF_OIII', 'EW50_Ha', 'EW16_Ha', 'EW84_Ha', 'F_Ha', 'eF_Ha']
row token counts: [(25, 12814)]
-> 12814 rows



In [70]:
NULLS = ['', '--', '---', 'nan', 'NaN', 'NULL', 'None', 'INDEF', '*']

def force_numeric(df, cols, sentinels=(-99, -999, -99.0, -999.0)):
    for c in cols:
        s = df[c].astype(str).str.strip().replace(NULLS, np.nan)
        df[c] = pd.to_numeric(s, errors='coerce').replace(list(sentinels), np.nan)
    return df

el = force_numeric(el, [c for c in el.columns if c != 'ID_g'])
z  = force_numeric(z,  ['RA','DEC','mag','ID_C20','bf','zp_lp','zp_ez','zg','qf_g','qf_gz'])

In [71]:
zt  = Table.from_pandas(z)
elt = Table.from_pandas(el)

zt['RA'].unit = zt['DEC'].unit = u.deg
elt['RA'].unit = elt['DEC'].unit = u.deg
for c in elt.colnames:
    if c.startswith(('EW50_', 'EW16_', 'EW84_')):
        elt[c].unit = u.AA          # rest-frame

# drop el-cat's duplicates of z-cat columns before joining
dup = ('RA', 'DEC', 'mag', 'z_g')
cat = join(zt, elt[[c for c in elt.colnames if c not in dup]],
           keys='ID_g', join_type='left')

print(f"z-cat {len(zt)}  el-cat {len(elt)}  joined {len(cat)}")
print("z-cat IDs missing from el-cat:", len(set(zt['ID_g']) - set(elt['ID_g'])))
print("el-cat IDs missing from z-cat:", len(set(elt['ID_g']) - set(zt['ID_g'])))
print("duplicate ID_g in z-cat:", len(zt) - len(set(zt['ID_g'])))

# sanity-check the flux units — grizli usually writes 1e-17 cgs
d = cat['F_OIII'][np.isfinite(cat['F_OIII'])]
print("median |F_OIII| =", np.median(np.abs(d)))

z-cat 12812  el-cat 12814  joined 12812
z-cat IDs missing from el-cat: 0
el-cat IDs missing from z-cat: 2
duplicate ID_g in z-cat: 0
median |F_OIII| = 5.50315e-17


/opt/conda/lib/python3.12/site-packages/numpy/core/fromnumeric.py:771: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedColumn.
  a.partition(kth, axis=axis, kind=kind, order=order)


In [72]:
zpath = 'catalogs/hlsp_hst-hyperion_hst_wfc3_hyperion_f140w-f160w-g141_v1_z-cat.txt'

raw = Path(zpath).read_text(errors="replace").splitlines()
hashed = [l for l in raw if l.lstrip().startswith("#")]
print(len(raw), "lines,", len(hashed), "hashed")
for l in hashed[-4:]: print(repr(l[:120]))

12813 lines, 1 hashed
'#  RA        DEC       mag      ID_C20  bf  zp_lp     zp_ez    ID_g                      zg    qf_g qf_gz  spec_com'


In [73]:
FEATURES = {'Ha','SII','OIII','Hb','Hg','Hd','NeIII','OII','PaB','cont_break'}
MODS   = ('marg_', 'wide_', 'ex_')
PA_RE  = re.compile(r'^PA_\d+_')

def _atoms(com):
    if com is None or str(com).strip() in ('', '--', 'nan'):
        return []
    out = []
    for tok in str(com).split():
        for a in tok.split('/'):
            a = a.strip().rstrip('!')
            if a:
                out.append(a)
    return out

def _strip_mods(a):
    a, mods = PA_RE.sub('', a), []
    hit = True
    while hit:
        hit = False
        for m in MODS:
            if a.startswith(m):
                mods.append(m.rstrip('_')); a = a[len(m):]; hit = True
    return a, mods

def parse_comment(com):
    feats, mods_seen, plain = set(), set(), set()
    for a in _atoms(com):
        base, mods = _strip_mods(a)
        mods_seen |= set(mods)
        (feats if base in FEATURES else plain).add(base)
        if base in FEATURES and mods:
            for m in mods:
                mods_seen.add(f'{m}:{base}')
    s = str(com or '')
    return dict(
        n_feat     = len(feats),
        feats      = '/'.join(sorted(feats)),
        agn_type1  = any(m.startswith('wide') for m in mods_seen),   # >100 A -> type-1
        unusable   = 'su' in plain,
        cw         = any(p.startswith('cw') for p in plain),
        cw_full    = 'cw_full' in plain,
        cutoff     = any(p.startswith('cut_') for p in plain),
        supser     = any(p.startswith('supser') for p in plain),
        cont_bad   = any(p.startswith(('oc', 'uc')) for p in plain) or 'ss' in plain,
        unfit_line = any(p.startswith('ef_') for p in plain),
        z_override = 'zc' in plain,
        emphatic   = '!' in s,
        pa_only    = 'PA_' in s,
    )

pc = [parse_comment(c) for c in cat['spec_com']]
for k in pc[0]:
    col = np.array([p[k] for p in pc])
    cat[k] = col

In [74]:
LYA_LO, LYA_HI = 1.879, 3.524     # Lya in VIRUS 3500-5500 A
OIII_HI        = 2.295            # [OIII]5007 leaves G141 at 1.65 um
OII_LO         = 1.951            # [OII]3727 enters G141 at 1.10 um

zg = cat['zg']
steps = [
    ("all",                     np.ones(len(cat), bool)),
    ("qf_gz >= 3",              cat['qf_gz'] >= 3),
    ("qf_gz >= 4",              cat['qf_gz'] >= 4),
    ("qf_gz == 5",              cat['qf_gz'] == 5),
    ("  + qf_g >= 3",           (cat['qf_gz'] == 5) & (cat['qf_g'] >= 3)),
    ("  + unblended",           (cat['qf_gz'] == 5) & (cat['qf_g'] >= 3) & (cat['bf'] == 0)),
    ("  + clean comments",      (cat['qf_gz'] == 5) & (cat['qf_g'] >= 3) & (cat['bf'] == 0)
                                 & ~cat['unusable'] & ~cat['cw_full'] & ~cat['supser']),
]
base = steps[-1][1]
steps += [
    ("  + in Lya window",       base & (zg > LYA_LO) & (zg < LYA_HI)),
    ("     of which [OIII]",    base & (zg > LYA_LO) & (zg < OIII_HI)),
    ("     of which O32",       base & (zg > OII_LO) & (zg < OIII_HI)),
    ("     of which OII-only",  base & (zg > OIII_HI) & (zg < LYA_HI)),
    ("  + >=2 features",        base & (zg > LYA_LO) & (zg < LYA_HI) & (cat['n_feat'] >= 2)),
    ("  + not type-1 AGN",      base & (zg > LYA_LO) & (zg < LYA_HI) & ~cat['agn_type1']),
]
for name, m in steps:
    print(f"{name:26s} {m.sum():6d}")

# where the single-feature wall sits
import collections
for lo, hi in [(1.88,2.10),(2.10,2.30),(2.30,2.60),(2.60,3.00),(3.00,3.52)]:
    m = base & (zg >= lo) & (zg < hi)
    print(f"{lo:.2f}-{hi:.2f}  N={m.sum():4d}  "
          f"median n_feat={np.median(cat['n_feat'][m]) if m.sum() else np.nan}")

all                         12812
qf_gz >= 3                   5630
qf_gz >= 4                   1594
qf_gz == 5                    654
  + qf_g >= 3                 650
  + unblended                 646
  + clean comments            640
  + in Lya window              83
     of which [OIII]           70
     of which O32              55
     of which OII-only         13
  + >=2 features               60
  + not type-1 AGN             82
1.88-2.10  N=  49  median n_feat=3.0
2.10-2.30  N=  23  median n_feat=2.0
2.30-2.60  N=  10  median n_feat=2.0
2.60-3.00  N=   1  median n_feat=3.0
3.00-3.52  N=   0  median n_feat=nan


In [78]:
import collections, numpy as np, astropy.units as u
from astropy.coordinates import SkyCoord

def hyperion_sample(cat, qf_min, zlo=1.879, zhi=3.524, strict=True):
    m = (cat['qf_gz'] >= qf_min) & (cat['zg'] > zlo) & (cat['zg'] < zhi)
    if strict:
        m &= (cat['qf_g'] >= 3) & (cat['bf'] == 0)
        m &= ~cat['unusable'] & ~cat['cw_full'] & ~cat['supser']
    return cat[m]

bow_c = bowman[bowman['field'] == 'cosmos']
c_bow = SkyCoord(np.asarray(bow_c['ra'],  float)*u.deg,
                 np.asarray(bow_c['dec'], float)*u.deg)
print(f"bowman: {len(bowman)} total, {len(bow_c)} COSMOS, "
      f"z {bowman['z'].min():.2f}-{bowman['z'].max():.2f}")

print(collections.Counter(np.asarray(bowman['field'])))

for qf in (3, 4, 5):
    s   = hyperion_sample(cat, qf)
    c_s = SkyCoord(np.asarray(s['RA'], float)*u.deg,
                   np.asarray(s['DEC'], float)*u.deg)
    idx, d2d, _ = c_s.match_to_catalog_sky(c_bow)
    hit = d2d < 0.5*u.arcsec
    newp = collections.Counter(i.rsplit('_', 1)[0]
                               for i in np.asarray(s['ID_g'])[~hit])
    print(f"qf_gz>={qf}: N={len(s):4d}  in bowman={int(hit.sum()):4d}  "
          f"new={int((~hit).sum()):4d}   from {newp.most_common(5)}")

bowman: 759 total, 289 COSMOS, z 1.90-2.35
Counter({'aegis': 470, 'cosmos': 289})
qf_gz>=3: N= 895  in bowman= 138  new= 757   from [('hyperion_peak1', 378), ('hyperion_peak2n3n7', 209), ('hyperion_peak4n5n6', 142), ('hyperion_peak4extra', 28)]
qf_gz>=4: N= 208  in bowman=  65  new= 143   from [('hyperion_peak1', 72), ('hyperion_peak2n3n7', 32), ('hyperion_peak4n5n6', 23), ('hyperion_peak4extra', 16)]
qf_gz>=5: N=  83  in bowman=  35  new=  48   from [('hyperion_peak1', 21), ('hyperion_peak2n3n7', 13), ('hyperion_peak4n5n6', 9), ('hyperion_peak4extra', 5)]


In [77]:
s   = hyperion_sample(cat, 4)
c_s = SkyCoord(np.asarray(s['RA'], float)*u.deg,
               np.asarray(s['DEC'], float)*u.deg)
idx, d2d, _ = c_s.match_to_catalog_sky(c_bow)
hit = d2d < 0.5*u.arcsec

zh, zb = np.asarray(s['zg'])[hit], np.asarray(bow_c['z'])[idx[hit]]
dz  = (zh - zb) / (1 + zb)
med = np.median(dz); sig = 1.4826*np.median(np.abs(dz - med)); zm = np.median(zb)
print(f"N={int(hit.sum())}  median={med:+.5f} ({med*2.998e5:+.0f} km/s)  "
      f"sigma_NMAD={sig:.5f} ({sig*2.998e5:.0f} km/s)")
print(f"  at Lya z={zm:.2f}: {1215.67*(1+zm)*sig:.1f} A scatter, "
      f"{1215.67*(1+zm)*abs(med):.1f} A systematic")
print("  outliers |dz|/(1+z)>0.004:", int((np.abs(dz) > 0.004).sum()))

# settles the F_OIII unit AND cross-checks the two reductions
fh = np.asarray(s['F_OIII'], float)[hit]
fb = np.asarray(bow_c['OIII_FLUX'], float)[idx[hit]]
g  = np.isfinite(fh) & np.isfinite(fb) & (fh > 0) & (fb > 0)
print(f"median F_OIII / OIII_FLUX = {np.median(fh[g]/fb[g]):.3g}  (N={g.sum()})")

N=65  median=+0.00009 (+28 km/s)  sigma_NMAD=0.00110 (330 km/s)
  at Lya z=2.09: 4.1 A scatter, 0.4 A systematic
  outliers |dz|/(1+z)>0.004: 1
median F_OIII / OIII_FLUX = 1.47  (N=65)


In [79]:
s = hyperion_sample(cat, 4)
c_s = SkyCoord(np.asarray(s['RA'],float)*u.deg, np.asarray(s['DEC'],float)*u.deg)
idx, d2d, _ = c_s.match_to_catalog_sky(c_bow); hit = d2d < 0.5*u.arcsec
print("MATCHED prefixes:", collections.Counter(
      i.rsplit('_',1)[0] for i in np.asarray(s['ID_g'])[hit]).most_common())

MATCHED prefixes: [('hyperion_peak1', 65)]


In [80]:
from scipy.spatial import Delaunay
hull = Delaunay(np.column_stack([np.asarray(bow_c['ra'],float),
                                 np.asarray(bow_c['dec'],float)]))
for qf in (3, 4, 5):
    s = hyperion_sample(cat, qf)
    P = np.column_stack([np.asarray(s['RA'],float), np.asarray(s['DEC'],float)])
    ins = hull.find_simplex(P) >= 0
    print(f"qf>={qf}: N={len(s):4d}  inside 3D-HST COSMOS={int(ins.sum()):4d}  "
          f"new sky={int((~ins).sum()):4d}")

qf>=3: N= 895  inside 3D-HST COSMOS= 465  new sky= 430
qf>=4: N= 208  inside 3D-HST COSMOS= 126  new sky=  82
qf>=5: N=  83  inside 3D-HST COSMOS=  50  new sky=  33


In [86]:
mos = Table.read("catalogs/lya_halo_catalog_cuts/lya_mosdef_catalog_KEEP.fits")
c_mos = SkyCoord(np.asarray(mos['RA'],float)*u.deg,      # adjust col names
                 np.asarray(mos['DEC'],float)*u.deg)

for name, tab, racol, deccol, zcol in [
        ("bowman",   bowman, 'ra', 'dec', 'z'),
        ("hyperion", hyperion_sample(cat, 4), 'RA', 'DEC', 'zg')]:
    c = SkyCoord(np.asarray(tab[racol],float)*u.deg,
                 np.asarray(tab[deccol],float)*u.deg)
    idx, d2d, _ = c.match_to_catalog_sky(c_mos); hit = d2d < 1*u.arcsec
    zg, zs = np.asarray(tab[zcol])[hit], np.asarray(mos['z'])[idx[hit]]
    dz = (zg - zs)/(1+zs); med = np.median(dz)
    sig = 1.4826*np.median(np.abs(dz-med)); zm = np.median(zs)
    print(f"{name:9s} N={int(hit.sum()):4d}  median={med*2.998e5:+6.0f} km/s  "
          f"sigma={sig*2.998e5:5.0f} km/s  -> {1215.67*(1+zm)*sig:.1f} A, "
          f"{1215.67*(1+zm)*abs(med):.1f} A sys")

bowman    N=  81  median=   +86 km/s  sigma=  391 km/s  -> 5.0 A, 1.1 A sys
hyperion  N=  18  median=   +45 km/s  sigma=  456 km/s  -> 5.9 A, 0.6 A sys


In [89]:
def nmad(d):
    m = np.median(d)
    return m, 1.4826 * np.median(np.abs(d - m))

def match_to_mosdef(tab, racol, deccol, zcol, radius=1*u.arcsec):
    c = SkyCoord(np.asarray(tab[racol], float)*u.deg,
                 np.asarray(tab[deccol], float)*u.deg)
    idx, d2d, _ = c.match_to_catalog_sky(c_mos)
    hit = d2d < radius
    sub = tab[hit]
    zs  = np.asarray(mos['z'], float)[idx[hit]]
    dz  = (np.asarray(sub[zcol], float) - zs) / (1 + zs)
    return sub, zs, dz

C = 2.998e5

def scatter_binned(v, dz, zs, edges, label, cumulative=False):
    print(f"\n-- {label} --")
    for i, lo in enumerate(edges[:-1]):
        hi = edges[i+1]
        k = np.isfinite(v) & ((v >= lo) if cumulative else ((v >= lo) & (v < hi)))
        tag = f">{lo:g}" if cumulative else f"{lo:g}-{hi:g}"
        if k.sum() < 5:
            print(f"  {tag:>12s}  N={k.sum():3d}   (too few)"); continue
        m, s = nmad(dz[k]); zm = np.median(zs[k])
        print(f"  {tag:>12s}  N={k.sum():3d}  med={m*C:+5.0f}  "
              f"sig={s*C:4.0f} +/-{s*C/np.sqrt(2*k.sum()):3.0f} km/s  "
              f"({1215.67*(1+zm)*s:.1f} A at Lya)")

# ---- bowman ----------------------------------------------------------------
bsub, bzs, bdz = match_to_mosdef(bowman, 'ra', 'dec', 'z')
m, s = nmad(bdz)
print(f"bowman x MOSDEF: N={len(bsub)}  med={m*C:+.0f}  sig={s*C:.0f} km/s")

err  = np.asarray(bsub['OIII_ERR'], float)
bsnr = np.where(err > 0, np.asarray(bsub['OIII_FLUX'], float) / err, np.nan)

scatter_binned(bsnr, bdz, bzs, [0, 5, 10, 20, 40], "bowman: OIII S/N", cumulative=True)
scatter_binned(bzs,  bdz, bzs, [1.90, 2.00, 2.10, 2.20, 2.35], "bowman: redshift")
scatter_binned(np.asarray(bsub['OIII_EW'], float), bdz, bzs,
               [0, 100, 250, 500, 1e4], "bowman: OIII_EW (A)")

# ---- hyperion (N~18, expect nothing conclusive) -----------------------------
hs = hyperion_sample(cat, 4)
hsub, hzs, hdz = match_to_mosdef(hs, 'RA', 'DEC', 'zg')
m, s = nmad(hdz)
print(f"\nhyperion x MOSDEF: N={len(hsub)}  med={m*C:+.0f}  sig={s*C:.0f} km/s")
herr  = np.asarray(hsub['eF_OIII'], float)
hsnr  = np.where(herr > 0, np.asarray(hsub['F_OIII'], float) / herr, np.nan)
scatter_binned(hsnr, hdz, hzs, [0, 10, 20], "hyperion: OIII S/N", cumulative=True)

bowman x MOSDEF: N=81  med=+86  sig=391 km/s

-- bowman: OIII S/N --
            >0  N= 81  med=  +86  sig= 391 +/- 31 km/s  (5.0 A at Lya)
            >5  N= 63  med=  +62  sig= 315 +/- 28 km/s  (4.1 A at Lya)
           >10  N= 38  med=  +61  sig= 315 +/- 36 km/s  (4.1 A at Lya)
           >20  N= 14  med=  +85  sig= 128 +/- 24 km/s  (1.7 A at Lya)

-- bowman: redshift --
         1.9-2  N=  1   (too few)
         2-2.1  N= 10  med=  +33  sig= 340 +/- 76 km/s  (4.3 A at Lya)
       2.1-2.2  N= 35  med=  +86  sig= 310 +/- 37 km/s  (4.0 A at Lya)
      2.2-2.35  N= 35  med=  +97  sig= 485 +/- 58 km/s  (6.4 A at Lya)

-- bowman: OIII_EW (A) --
         0-100  N= 31  med= +133  sig= 571 +/- 73 km/s  (7.4 A at Lya)
       100-250  N= 39  med=  +60  sig= 366 +/- 41 km/s  (4.7 A at Lya)
       250-500  N=  8  med=  +68  sig= 107 +/- 27 km/s  (1.4 A at Lya)
     500-10000  N=  3   (too few)

hyperion x MOSDEF: N=18  med=+45  sig=456 km/s

-- hyperion: OIII S/N --
            >0  N= 15  med= 

# Hyperion Cluster

In [90]:
PEAKS = {  # Cucciati+2018, via Forrest+2025 Table 4
    'Theia':  (150.0937, 2.4049, 2.468), 'Eos':    (149.9765, 2.1124, 2.426),
    'Helios': (149.9996, 2.2537, 2.444), 'Selene': (150.2556, 2.3423, 2.469),
    'peak5':  (150.2293, 2.3381, 2.507), 'peak6':  (150.3316, 2.2427, 2.492),
    'peak7':  (149.9581, 2.2187, 2.423),
}

def hyperion_structure(cat, qf_min=3, zlo=2.35, zhi=2.65,
                       peak_radius=3*u.arcmin, dz_peak=0.025):
    """Grism members of the Hyperion structure. qf_min=3 + no extra cuts
    reproduces the paper's ~192-196."""
    s = cat[(cat['qf_gz'] >= qf_min) & (cat['zg'] > zlo) & (cat['zg'] < zhi)]
    c = SkyCoord(np.asarray(s['RA'], float)*u.deg,
                 np.asarray(s['DEC'], float)*u.deg)
    peak = np.array(['field'] * len(s), dtype='U8')
    for name, (pra, pdec, pz) in PEAKS.items():
        near = (c.separation(SkyCoord(pra*u.deg, pdec*u.deg)) < peak_radius) \
               & (np.abs(np.asarray(s['zg'], float) - pz) < dz_peak)
        peak[near & (peak == 'field')] = name
    s['peak'] = peak
    return s

st = hyperion_structure(cat)
print(len(st), "grism members in 2.35 < z < 2.65   (paper: 192-196)")
print(collections.Counter(np.asarray(st['peak'])))
print(collections.Counter(np.asarray(st['qf_gz'])))

196 grism members in 2.35 < z < 2.65   (paper: 192-196)
Counter({'field': 148, 'Selene': 11, 'peak5': 11, 'Theia': 9, 'Eos': 7, 'Helios': 6, 'peak7': 2, 'peak6': 2})
Counter({3: 166, 4: 24, 5: 6})
